In [ ]:
import modules.data_scvi as d 
import modules.utils as u
from pathlib import Path

#### init ####
dataset_dir = Path('/home/mv18gs/Documents/GitHub/pathway_model/datasets/')
device = u.Devices().auto_set_device()#drop=['cuda:4','cuda:7'])

('cuda:4', 'NVIDIA A100-SXM4-80GB', 81043)
('cuda:2', 'NVIDIA A100-SXM4-80GB', 81041)
('cuda:3', 'NVIDIA A100-SXM4-80GB', 81041)
('cuda:6', 'NVIDIA A100-SXM4-80GB', 81041)
('cuda:7', 'NVIDIA A100-SXM4-80GB', 81041)
('cuda:0', 'NVIDIA A100-SXM4-80GB', 78933)
('cuda:5', 'NVIDIA A100-SXM4-80GB', 74623)
('cuda:1', 'NVIDIA A100-SXM4-80GB', 66795)

# #### Device() ####
# device = cuda:4



---

In [2]:
from modules.layers import AttentionSetPooling
from modules.loss import MultiLoss, NBLoss, KLDLoss
from modules.model import CountsAutoencoder, NBGLM
from modules.norm import LogCounts
from modules.train import Experiment, grid
from modules.trainers import ReconstrTrainer

from functools import partial
import torch.nn as nn

In [ ]:
#### data ####
tcga = {
    'BRCA': d.TCGA(
        tcga_project = 'BRCA',
        tcga_dir = dataset_dir/'tcga',
        subtype_col = 'paper_BRCA_Subtype_PAM50',
        drop = ['Normal', 'Primary Tumor', 'Metastatic'],
        gene_name_path = dataset_dir/'other'/'name2ensg.csv',
        keep_noname = False,
    ),

    'GBM': d.TCGA(
        tcga_project = 'GBM',
        tcga_dir = dataset_dir/'tcga',
        subtype_col = 'paper_Original.Subtype',
        drop = ['Primary Tumor', 'Recurrent Tumor', 'Solid Tissue Normal'],
        gene_name_path = dataset_dir/'other'/'name2ensg.csv',
        keep_noname = False,
    ),

    'LGG': d.TCGA(
        tcga_project = 'LGG',
        tcga_dir = dataset_dir/'tcga',
        subtype_col = 'paper_Original.Subtype',
        drop = ['Primary Tumor', 'Recurrent Tumor'],
        gene_name_path = dataset_dir/'other'/'name2ensg.csv',
        keep_noname = False,
    ),
}

In [ ]:
def run_expt(counts_name:str, counts_data:d.TCGA, num_trials:int=5):
    # data
    kegg = d.KEGG(
        relation_filepath=dataset_dir/'other'/'relation_ohe.csv', 
        counts_data=counts_data,
    )
    dataset = d.GraphDataset(counts_data, kegg, kegg)

    # models
    glm = NBGLM(dataset)
    vae = CountsAutoencoder(
        dataset=dataset,
        embed_dim = 128,
        method='node',


        norm_class = LogCounts,
        encoder_class = nn.Linear,
        pooling_class = AttentionSetPooling,
        mlp = False,
        variational = True,

        hidden_dims = 1,
        act_fn = nn.ReLU, 
        norm_fn = 'layer', 
        end_fn = False,

        norm_kwargs = {'libnorm':True, 'znorm':True, 'learnable':True,}
    )
    pw = CountsAutoencoder(
        dataset=dataset,
        embed_dim = 128,
        method='set',


        norm_class = LogCounts,
        encoder_class = nn.Linear,
        pooling_class = AttentionSetPooling,
        mlp = False,
        variational = True,

        hidden_dims = 1,
        act_fn = nn.ReLU, 
        norm_fn = 'layer', 
        end_fn = False,

        norm_kwargs = {'libnorm':True, 'znorm':True, 'learnable':True,}
    )

    # trainer
    glm_trainer = ReconstrTrainer(
        lr=1e-3, 

        trainer_norm_class=LogCounts,
        early_stop=True,
        stop_metric='mae',

        kw_keys={'x':'x', 'mu':'x_pred', 'theta':'theta'},
        metric_keys={'pred':'mu', 'target':'x'},
        loss_class=NBLoss,
    )

    vae_trainer = ReconstrTrainer(
        lr=1e-3, 

        trainer_norm_class=LogCounts,
        early_stop=True,
        stop_metric='mae',

        kw_keys=('x','x_pred','theta','z_mu','z_logvar'),
        metric_keys={'pred':'x_pred', 'target':'x'},
        
        loss_class=MultiLoss,
        loss_kwargs = {
        'loss_classes': [NBLoss, KLDLoss],
        'loss_weights': [1.0, 1e-4],
        'loss_inputs': [
            {'x':'x', 'mu':'x_pred', 'theta':'theta'},
            {'mu':'z_mu', 'logvar':'z_logvar'}
        ]
    },
    )
    
    # expt
    expt = Experiment(
        num_trials=num_trials, # scVI: 5 trials
        num_epochs=300, # scVI: 200 epochs
        dataset=dataset,
        device=device,
        batch_size=128
    )

    expt.add_config(
        name='glm',
        model=glm,
        trainer=glm_trainer,
    )

    expt.add_config(
        name='gene-vae',
        model=vae,
        trainer=vae_trainer,
    )

    expt.add_config(
        name='pathway-vae',
        model=pw,
        trainer=vae_trainer,
    )

    expt.run_experiment(
        f'benchmark_1_nbreconstr_tcga_{counts_name}',
        report_metrics=['loss','kld','nb','mae', 'accuracy', 'precision', 'f1'],
        # save_model=True,
        save_csv=True,
        save_params=True,
        save_values=True,
        verbose=False,
    )

In [ ]:
for k,v in tcga.items():
    run_expt(k, v, num_trials=5)

for k,v in tcga.items():
    run_expt(k, v, num_trials=25)

# #### KEGG() ####
# _orig_kwargs             5                        dict
# relation                 (75939, 19)              DataFrame
# ensg                     4373                     list
# pathway_labels           305                      list
# edge_index               (2, 32464)               Tensor (cuda:4)
# edge_attr                (32464, 16)              Tensor (cuda:4)
# edge_labels              16                       list
# pathway_index            (4373, 305)              Tensor (cuda:4)

# #### TCGA() ####
# _orig_kwargs             9                        dict
# counts_path              PosixPath
# metadata_path            PosixPath
# gene_name_path           PosixPath
# metadata_complete        (1231, 93)               DataFrame
# metadata                 (1172, 2)                DataFrame
# y                        (1172,)                  Tensor (cuda:4)
# y_labels                 5                        list
# ensgv                    (60660, 3)             

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/3 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]

...

  0%|          | 0/300 [00:00<?, ?it/s]

  0%|          | 0/3 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]